In [ ]:
import os, json, pandas as pd, numpy as np, joblib, matplotlib.pyplot as plt
import thermoift.PLOT_SETTINGS as ps
from thermoift import MLPostprocessing, plot_correlation_heatmap

In [ ]:
PLOT_FOLDER = "TabPFN_interfacial_thickness_OUTPUTS"
target      = "interfacial_thickness"

In [ ]:
with open(os.path.join(PLOT_FOLDER, f"TabPFN_{target}_metrics.json")) as f:
    metrics = json.load(f)

features     = metrics["features"]
cv_r2_mean   = metrics["cv_r2_mean"]
cv_rmse_mean = metrics["cv_rmse_mean"]
cv_mae_mean  = metrics["cv_mae_mean"]

preds_df = pd.read_csv(os.path.join(PLOT_FOLDER, f"TabPFN_{target}_predictions.csv"))

print(f"Loaded artifacts from {PLOT_FOLDER}/")
print(f"Features : {features}")
print(f"CV R²    : {cv_r2_mean:.6f}  |  CV RMSE: {cv_rmse_mean:.6f}  |  CV MAE: {cv_mae_mean:.6f}")

In [ ]:
train_rows = preds_df[preds_df["split"] == "train"]
test_rows  = preds_df[preds_df["split"] == "test"]
val_rows   = preds_df[preds_df["split"] == "val"]

y_train      = pd.Series(train_rows["actual"].values, name=target)
y_train_pred = train_rows["predicted"].values
y_test       = pd.Series(test_rows["actual"].values, index=test_rows["idx"].values, name=target)
y_test_pred  = test_rows["predicted"].values
y_val        = pd.Series(val_rows["actual"].values, name=target)
y_val_pred   = val_rows["predicted"].values

post = MLPostprocessing(
    y_true=y_test,
    y_pred=y_test_pred,
    target=target,
    feature_names=features,
    datasets={
        "train": (y_train, y_train_pred),
        "test":  (y_test,  y_test_pred),
        "val":   (y_val,   y_val_pred),
    },
)

In [ ]:
df = pd.read_csv("../../interfacial_results_dataset_A4.csv")
plot_correlation_heatmap(df, features, target, save_path=f"TabPFN_{target}_correlation", folder=PLOT_FOLDER)

In [ ]:
post.plot_parity(model_name="TabPFN", save_path=f"TabPFN_{target}_parity_plot", folder=PLOT_FOLDER,
                 cv_r2=cv_r2_mean, cv_rmse=cv_rmse_mean, cv_mae=cv_mae_mean)

In [ ]:
post.plot_residual_distribution(save_path=f"TabPFN_{target}_residual_distribution", folder=PLOT_FOLDER,
                                cv_r2=cv_r2_mean, cv_rmse=cv_rmse_mean, cv_mae=cv_mae_mean)

In [ ]:
post.plot_residual_vs_predicted(save_path=f"TabPFN_{target}_residual_vs_predicted", folder=PLOT_FOLDER,
                                cv_r2=cv_r2_mean, cv_rmse=cv_rmse_mean, cv_mae=cv_mae_mean)

In [ ]:
post.print_summary()